# PatchCore Grid Baseline (Colab)

MVTec-AD **grid** 카테고리 단일 베이스라인. 현송 담당.

## 플로우
1. Drive 마운트 + `grid_split.zip` 업로드 (또는 이미 풀린 `grid_split/` 폴더)
2. PatchCore 학습 (ResNet18 → Wide-ResNet50-2 순서)
3. `val` / `test` 평가
4. MLflow로 실험 기록 (Drive에 저장 → 세션 끊겨도 유지)
5. 모델/플롯 저장 + 결과 비교 테이블

## 사전 준비
Drive에 아래 중 하나로 업로드:

**(A) zip 업로드** (추천) — `/content/drive/MyDrive/SWcapstone/grid_split.zip`

**(B) 폴더 업로드** — `/content/drive/MyDrive/SWcapstone/grid_split/` 아래에:
```
grid_split/
├── train/normal/              (185장)
├── val/
│   ├── normal/                (40장)
│   └── anomaly/{bent,broken,glue,metal_contamination,thread}/
└── test/
    ├── normal/                (39장)
    └── anomaly/{bent,broken,glue,metal_contamination,thread}/
```

CSV 없이 **폴더 구조 자체로 split/label을 인식**. `models/`, `reports/`, `mlruns/` 폴더는 자동 생성.

## 1. 환경 세팅

In [1]:
# Colab 기본: torch, torchvision, scipy, scikit-learn, matplotlib 이미 설치됨
!pip install -q mlflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 68.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 82.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 33.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.2/131.2 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.5/838.5 kB 46.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. 경로 & 하이퍼파라미터
경로만 수정하면 다른 카테고리/위치에도 재사용 가능.

In [3]:
from pathlib import Path
import torch

# ---- 경로 (필요시 수정) ----
PROJECT_ROOT = Path('/content/drive/MyDrive/SWcapstone')
DATA_ROOT    = PROJECT_ROOT / 'grid_split'
ZIP_PATH     = PROJECT_ROOT / 'grid_split.zip'
MODELS_DIR   = PROJECT_ROOT / 'models'
REPORTS_DIR  = PROJECT_ROOT / 'reports'
MLRUNS_DIR   = PROJECT_ROOT / 'mlruns'

# ---- 하이퍼파라미터 ----
SEED            = 42
INPUT_SIZE      = 256       # 224 → 256 (미세 결함 가시성↑)
BATCH_SIZE      = 32
NUM_WORKERS     = 2
CORESET_RATIO   = 0.10      # 0.25 → 0.10 (greedy coreset은 작게도 충분)
NUM_NEIGHBORS   = 1         # 9 → 1 (PatchCore 논문 기준, anomaly 신호 희석 방지)
POOL_KERNEL     = 3         # locally-aware feature pooling (핵심 수정)
CORESET_METHOD  = 'greedy'  # 'random' | 'greedy'
CORESET_MAX_CANDIDATES = 50000   # greedy 입력 pool 상한 (메모리/속도)

# ---- 디바이스 ----
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

Device: cuda
GPU: Tesla T4


## 3. 데이터 준비
`grid_split/` 가 이미 풀려있으면 그대로 사용, 없으면 `grid_split.zip` 을 풂.

In [4]:
import zipfile

if not DATA_ROOT.exists():
    assert ZIP_PATH.exists(), (
        f'neither {DATA_ROOT} nor {ZIP_PATH} exists. '
        f'Drive에 grid_split/ 폴더 또는 grid_split.zip 을 업로드해주세요.'
    )
    print(f'Extracting {ZIP_PATH} ...')
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(PROJECT_ROOT)
    print(f'Extracted -> {DATA_ROOT}')
else:
    print(f'Using existing folder: {DATA_ROOT}')

# 구조 검증
expected = [
    'train/normal',
    'val/normal', 'val/anomaly',
    'test/normal', 'test/anomaly',
]
for rel in expected:
    p = DATA_ROOT / rel
    assert p.exists(), f'missing: {p}'
    print(f'OK  {rel}')

# 출력 폴더 생성
for d in [MODELS_DIR, REPORTS_DIR, MLRUNS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

Using existing folder: /content/drive/MyDrive/SWcapstone/grid_split
OK  train/normal
OK  val/normal
OK  val/anomaly
OK  test/normal
OK  test/anomaly


## 4. Imports & Seed

In [5]:
import os, random, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as T
from PIL import Image
from scipy.ndimage import gaussian_filter, zoom
from sklearn.metrics import (
    auc, average_precision_score, confusion_matrix, f1_score,
    precision_recall_curve, precision_score, recall_score, roc_auc_score, roc_curve,
)
from torch.utils.data import DataLoader, Dataset
import mlflow

warnings.filterwarnings('ignore', category=UserWarning)

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

def set_seed(seed: int) -> None:
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)

## 5. Dataset
폴더 구조를 walk 해서 이미지 목록 구성.
- `<split>/normal/*` → label 0 (normal)
- `<split>/anomaly/<category>/*` → label 1 (anomaly)

In [6]:
IMG_EXTS = {'.png', '.jpg', '.jpeg', '.bmp'}

class GridFolderDataset(Dataset):
    """grid_split/<split>/... 을 읽는 데이터셋."""
    def __init__(self, data_root, split, transform=None):
        self.split_dir = Path(data_root) / split
        self.transform = transform
        self.samples = []  # list of (path, label)

        # normal
        normal_dir = self.split_dir / 'normal'
        if normal_dir.exists():
            for p in sorted(normal_dir.iterdir()):
                if p.suffix.lower() in IMG_EXTS:
                    self.samples.append((p, 0))

        # anomaly (by category)
        anomaly_dir = self.split_dir / 'anomaly'
        if anomaly_dir.exists():
            for cat_dir in sorted(anomaly_dir.iterdir()):
                if not cat_dir.is_dir():
                    continue
                for p in sorted(cat_dir.iterdir()):
                    if p.suffix.lower() in IMG_EXTS:
                        self.samples.append((p, 1))

        assert len(self.samples) > 0, f'no images found in {self.split_dir}'

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, label, str(path)

def get_transforms(input_size=224):
    return T.Compose([
        T.Resize((input_size, input_size)),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])

# --- sanity check ---
for split in ['train', 'val', 'test']:
    ds = GridFolderDataset(DATA_ROOT, split)
    n_norm = sum(1 for _, y in ds.samples if y == 0)
    n_anom = sum(1 for _, y in ds.samples if y == 1)
    print(f'{split:5s}: total={len(ds):3d}  normal={n_norm}  anomaly={n_anom}')

train: total=185  normal=185  anomaly=0
val  : total= 67  normal=40  anomaly=27
test : total= 69  normal=39  anomaly=30


## 6. PatchCore Model
- `FeatureExtractor`: pretrained backbone의 `layer2`, `layer3` feature 추출
- `PatchCoreModel`: coreset 서브샘플링 + k-NN 거리 기반 anomaly score

In [7]:
class FeatureExtractor(nn.Module):
    def __init__(self, backbone_name='resnet18', device='cpu', pool_kernel=3):
        super().__init__()
        self.device = device
        self.pool_kernel = pool_kernel
        if backbone_name == 'resnet18':
            backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        elif backbone_name == 'wide_resnet50_2':
            backbone = models.wide_resnet50_2(weights=models.Wide_ResNet50_2_Weights.DEFAULT)
        else:
            raise ValueError(f'Unsupported backbone: {backbone_name}')
        self.layer_names = ['layer2', 'layer3']
        self.backbone = backbone.eval()
        self._features = {}
        for name in self.layer_names:
            layer = dict(self.backbone.named_children())[name]
            layer.register_forward_hook(self._make_hook(name))
        self.to(device)

    def _make_hook(self, name):
        def hook(_m, _i, out):
            self._features[name] = out
        return hook

    @torch.no_grad()
    def forward(self, x):
        self._features.clear()
        x = x.to(self.device)
        self.backbone(x)
        target_size = self._features[self.layer_names[0]].shape[2:]
        feats = []
        for name in self.layer_names:
            f = self._features[name]
            if f.shape[2:] != target_size:
                f = torch.nn.functional.interpolate(
                    f, size=target_size, mode='bilinear', align_corners=False
                )
            feats.append(f)
        out = torch.cat(feats, dim=1)
        # ---- Locally-aware patch features (PatchCore 논문 방식) ----
        if self.pool_kernel and self.pool_kernel > 1:
            pad = self.pool_kernel // 2
            out = torch.nn.functional.avg_pool2d(
                out, kernel_size=self.pool_kernel, stride=1, padding=pad
            )
        return out


def greedy_coreset(features, n_select, device='cpu', max_candidates=50000, seed=42):
    """Greedy furthest-point sampling for coreset construction.

    Args:
        features: (N, C) CPU tensor
        n_select: number of points to select
        device: GPU/CPU device for distance computation
        max_candidates: if N > this, randomly subsample first (memory/speed)
        seed: random seed for the pre-filter and initial point

    Returns:
        (n_select, C) CPU tensor of selected features
    """
    g = torch.Generator().manual_seed(seed)
    features = features.cpu()
    n_total = features.shape[0]

    # Pre-filter large pools
    if n_total > max_candidates:
        perm = torch.randperm(n_total, generator=g)[:max_candidates]
        features = features[perm]
        n_total = max_candidates

    n_select = min(n_select, n_total)
    f_gpu = features.to(device)

    # Start from a random point
    first_idx = int(torch.randint(0, n_total, (1,), generator=g).item())
    selected_mask = torch.zeros(n_total, dtype=torch.bool, device=device)
    selected_mask[first_idx] = True

    # Track squared distance to the nearest selected point so far
    diff = f_gpu - f_gpu[first_idx]
    min_sq = (diff * diff).sum(dim=1)

    for step in range(n_select - 1):
        # Furthest-from-selected point
        masked = min_sq.clone()
        masked[selected_mask] = -1.0
        new_idx = int(torch.argmax(masked).item())
        selected_mask[new_idx] = True

        # Update min distances incrementally
        diff = f_gpu - f_gpu[new_idx]
        new_sq = (diff * diff).sum(dim=1)
        min_sq = torch.minimum(min_sq, new_sq)

        if (step + 1) % 500 == 0:
            print(f'  [greedy_coreset] {step + 1}/{n_select - 1} selected')

    selected_idx = torch.where(selected_mask)[0].cpu()
    return features[selected_idx]


class PatchCoreModel:
    def __init__(self, backbone_name='resnet18', coreset_sampling_ratio=0.10,
                 num_neighbors=9, device='cpu', pool_kernel=3,
                 coreset_method='random', coreset_max_candidates=50000):
        self.backbone_name = backbone_name
        self.coreset_ratio = coreset_sampling_ratio
        self.num_neighbors = num_neighbors
        self.device = device
        self.coreset_method = coreset_method
        self.coreset_max_candidates = coreset_max_candidates
        self.extractor = FeatureExtractor(backbone_name, device, pool_kernel=pool_kernel)
        self.memory_bank = None

    def fit(self, dataloader):
        print('[PatchCore] Extracting features from normal images ...')
        all_patches = []
        for batch_idx, (imgs, _, _) in enumerate(dataloader):
            feats = self.extractor(imgs)
            B, C, H, W = feats.shape
            patches = feats.permute(0, 2, 3, 1).reshape(-1, C)
            all_patches.append(patches.cpu())
        all_patches_t = torch.cat(all_patches, dim=0)
        n_total = all_patches_t.shape[0]
        n_coreset = max(1, int(n_total * self.coreset_ratio))
        print(f'[PatchCore] total_patches={n_total}  target_coreset={n_coreset}  method={self.coreset_method}')

        if self.coreset_method == 'greedy':
            self.memory_bank = greedy_coreset(
                all_patches_t,
                n_select=n_coreset,
                device=self.device,
                max_candidates=self.coreset_max_candidates,
            )
        else:
            idx = torch.randperm(n_total)[:n_coreset]
            self.memory_bank = all_patches_t[idx]

        print(f'[PatchCore] final memory_bank={self.memory_bank.shape[0]}')

    @torch.no_grad()
    def predict(self, dataloader):
        assert self.memory_bank is not None, 'Call fit() first'
        memory = self.memory_bank.to(self.device)
        all_scores, all_labels, all_paths, all_heatmaps = [], [], [], []
        for imgs, labels, paths in dataloader:
            feats = self.extractor(imgs)
            B, C, H, W = feats.shape
            for i in range(B):
                patch = feats[i].permute(1, 2, 0).reshape(-1, C)
                dists = torch.cdist(patch.to(self.device), memory)
                knn_dists, _ = dists.topk(self.num_neighbors, largest=False, dim=1)
                patch_scores = knn_dists.mean(dim=1)
                all_scores.append(patch_scores.max().item())
                all_labels.append(labels[i].item())
                all_paths.append(paths[i])
                hmap = patch_scores.cpu().numpy().reshape(H, W)
                hmap = gaussian_filter(hmap, sigma=4)
                all_heatmaps.append(hmap)
        return np.array(all_scores), np.array(all_labels), all_paths, all_heatmaps

    def save(self, path):
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        torch.save({
            'backbone_name': self.backbone_name,
            'coreset_ratio': self.coreset_ratio,
            'num_neighbors': self.num_neighbors,
            'coreset_method': self.coreset_method,
            'memory_bank': self.memory_bank,
        }, path)
        print(f'[PatchCore] saved -> {path}')

## 7. Metrics & Plots

In [8]:
def compute_metrics(y_true, scores):
    auroc = roc_auc_score(y_true, scores)
    auprc = average_precision_score(y_true, scores)
    precisions, recalls, thresholds = precision_recall_curve(y_true, scores)
    f1s = 2 * precisions * recalls / (precisions + recalls + 1e-8)
    best = int(np.argmax(f1s))
    th = float(thresholds[min(best, len(thresholds) - 1)])
    y_pred = (scores >= th).astype(int)
    return {
        'auroc': auroc, 'auprc': auprc,
        'f1': f1_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'threshold': th,
    }, th

def plot_roc(y_true, scores, save_path):
    fpr, tpr, _ = roc_curve(y_true, scores); a = auc(fpr, tpr)
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.plot(fpr, tpr, lw=2, label=f'AUC={a:.4f}'); ax.plot([0,1],[0,1],'k--',lw=1)
    ax.set_xlabel('FPR'); ax.set_ylabel('TPR'); ax.set_title('ROC')
    ax.legend(loc='lower right'); fig.tight_layout()
    save_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(save_path, dpi=150); plt.close(fig)

def plot_pr(y_true, scores, save_path):
    prec, rec, _ = precision_recall_curve(y_true, scores)
    ap = average_precision_score(y_true, scores)
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.plot(rec, prec, lw=2, label=f'AP={ap:.4f}')
    ax.set_xlabel('Recall'); ax.set_ylabel('Precision'); ax.set_title('Precision-Recall')
    ax.legend(loc='lower left'); fig.tight_layout()
    save_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(save_path, dpi=150); plt.close(fig)

def plot_cm(y_true, y_pred, save_path):
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(5, 5))
    im = ax.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
    ax.set_title('Confusion Matrix'); fig.colorbar(im, ax=ax)
    classes = ['normal', 'anomaly']
    ax.set_xticks([0, 1]); ax.set_xticklabels(classes)
    ax.set_yticks([0, 1]); ax.set_yticklabels(classes)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                    color='white' if cm[i, j] > cm.max() / 2 else 'black')
    ax.set_ylabel('True'); ax.set_xlabel('Predicted'); fig.tight_layout()
    save_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(save_path, dpi=150); plt.close(fig)

def plot_heatmaps(paths, heatmaps, labels, scores, save_path, n=8):
    n = min(n, len(paths))
    if n == 0: return
    sorted_idx = np.argsort(scores)
    top = sorted_idx[-n // 2:].tolist()
    bot = sorted_idx[: n - n // 2].tolist()
    idxs = bot + top
    fig, axes = plt.subplots(2, n, figsize=(3 * n, 6))
    if n == 1:
        axes = np.array(axes).reshape(2, 1)
    for col, i in enumerate(idxs):
        img = Image.open(paths[i]).convert('RGB').resize((224, 224))
        img_np = np.array(img)
        hmap = heatmaps[i]
        hmap_r = zoom(hmap, (224 / hmap.shape[0], 224 / hmap.shape[1]))
        label_str = 'anomaly' if labels[i] == 1 else 'normal'
        axes[0, col].imshow(img_np)
        axes[0, col].set_title(f'{label_str}\ns={scores[i]:.3f}', fontsize=8)
        axes[0, col].axis('off')
        axes[1, col].imshow(img_np)
        axes[1, col].imshow(hmap_r, cmap='jet', alpha=0.5)
        axes[1, col].set_title('heatmap', fontsize=8)
        axes[1, col].axis('off')
    fig.suptitle('Sample Heatmap Overlays', fontsize=12)
    fig.tight_layout()
    save_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(save_path, dpi=150); plt.close(fig)

## 8. MLflow 트래킹 설정
트래킹 폴더를 Drive에 두어 세션이 끊겨도 run history가 남음.

In [9]:
tracking_uri = f'file://{MLRUNS_DIR}'
mlflow.set_tracking_uri(tracking_uri)
mlflow.set_experiment('grid_patchcore_baseline')
print(f'MLflow tracking URI: {tracking_uri}')

/usr/local/lib/python3.12/dist-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


MLflow tracking URI: file:///content/drive/MyDrive/SWcapstone/mlruns


## 9. 실험 실행 함수
한 번 실행하면 train 학습 → val 평가 → test 평가 → 모델 저장 → MLflow 로깅 모두 수행.

In [10]:
def run_experiment(run_tag, backbone_name):
    print('=' * 60)
    print(f'Running: {run_tag}  ({backbone_name})')
    print('=' * 60)
    set_seed(SEED)
    transform = get_transforms(INPUT_SIZE)

    with mlflow.start_run(run_name=run_tag):
        mlflow.log_params({
            'backbone': backbone_name,
            'seed': SEED,
            'input_size': INPUT_SIZE,
            'batch_size': BATCH_SIZE,
            'coreset_sampling_ratio': CORESET_RATIO,
            'coreset_method': CORESET_METHOD,
            'num_neighbors': NUM_NEIGHBORS,
            'pool_kernel': POOL_KERNEL,
            'device': DEVICE,
            'data_root': str(DATA_ROOT),
        })

        # ---- Train: fit memory bank on train/normal ----
        train_ds = GridFolderDataset(DATA_ROOT, 'train', transform)
        train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE,
                              shuffle=False, num_workers=NUM_WORKERS)
        print(f'Train images: {len(train_ds)}')
        model = PatchCoreModel(
            backbone_name=backbone_name,
            coreset_sampling_ratio=CORESET_RATIO,
            num_neighbors=NUM_NEIGHBORS,
            device=DEVICE,
            pool_kernel=POOL_KERNEL,
            coreset_method=CORESET_METHOD,
            coreset_max_candidates=CORESET_MAX_CANDIDATES,
        )
        model.fit(train_dl)
        mlflow.log_metric('train/num_images', len(train_ds))
        mlflow.log_metric('train/memory_bank_size', int(model.memory_bank.shape[0]))

        # ---- Evaluate on val and test ----
        results = {}
        for split_name in ['val', 'test']:
            print(f'\n-- evaluate {split_name} --')
            ds = GridFolderDataset(DATA_ROOT, split_name, transform)
            dl = DataLoader(ds, batch_size=BATCH_SIZE,
                            shuffle=False, num_workers=NUM_WORKERS)
            scores, labels, paths, heatmaps = model.predict(dl)

            # --- 스코어 분포 디버깅 (normal vs anomaly 분리도) ---
            normal_scores = scores[labels == 0]
            anomaly_scores = scores[labels == 1]
            print(f'  normal  scores: mean={normal_scores.mean():.3f}  '
                  f'std={normal_scores.std():.3f}  '
                  f'min={normal_scores.min():.3f}  max={normal_scores.max():.3f}')
            print(f'  anomaly scores: mean={anomaly_scores.mean():.3f}  '
                  f'std={anomaly_scores.std():.3f}  '
                  f'min={anomaly_scores.min():.3f}  max={anomaly_scores.max():.3f}')
            gap = anomaly_scores.mean() - normal_scores.mean()
            print(f'  mean gap (anomaly - normal): {gap:+.3f}  '
                  f"({'GOOD' if gap > 0 else 'BAD: inverted'})")

            metrics, th = compute_metrics(labels, scores)
            for k, v in metrics.items():
                print(f'  {k}: {v:.4f}')
                mlflow.log_metric(f'{split_name}/{k}', v)

            y_pred = (scores >= th).astype(int)
            prefix = f'{run_tag}_{split_name}'
            plot_roc(labels, scores, REPORTS_DIR / f'{prefix}_roc.png')
            plot_pr(labels, scores, REPORTS_DIR / f'{prefix}_pr.png')
            plot_cm(labels, y_pred, REPORTS_DIR / f'{prefix}_cm.png')
            plot_heatmaps(paths, heatmaps, labels, scores,
                          REPORTS_DIR / f'{prefix}_heatmaps.png')
            for suffix in ['roc', 'pr', 'cm', 'heatmaps']:
                p = REPORTS_DIR / f'{prefix}_{suffix}.png'
                if p.exists():
                    mlflow.log_artifact(str(p), artifact_path=f'plots/{split_name}')
            results[split_name] = metrics

        # ---- Save model ----
        model_path = MODELS_DIR / f'{run_tag}.pt'
        model.save(model_path)
        mlflow.log_artifact(str(model_path), artifact_path='models')

        print(f'\n[{run_tag}] done.')
        return results

## 10. Run 1 — PatchCore (ResNet18)
빠른 sanity check. grid에서 AUROC 0.95+ 나와야 정상.

In [11]:
results_r18 = run_experiment('grid_patchcore_r18', 'resnet18')

Running: grid_patchcore_r18  (resnet18)
Train images: 185
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 124MB/s]


[PatchCore] Extracting features from normal images ...
[PatchCore] total_patches=189440  target_coreset=18944  method=greedy
  [greedy_coreset] 500/18943 selected
  [greedy_coreset] 1000/18943 selected
  [greedy_coreset] 1500/18943 selected
  [greedy_coreset] 2000/18943 selected
  [greedy_coreset] 2500/18943 selected
  [greedy_coreset] 3000/18943 selected
  [greedy_coreset] 3500/18943 selected
  [greedy_coreset] 4000/18943 selected
  [greedy_coreset] 4500/18943 selected
  [greedy_coreset] 5000/18943 selected
  [greedy_coreset] 5500/18943 selected
  [greedy_coreset] 6000/18943 selected
  [greedy_coreset] 6500/18943 selected
  [greedy_coreset] 7000/18943 selected
  [greedy_coreset] 7500/18943 selected
  [greedy_coreset] 8000/18943 selected
  [greedy_coreset] 8500/18943 selected
  [greedy_coreset] 9000/18943 selected
  [greedy_coreset] 9500/18943 selected
  [greedy_coreset] 10000/18943 selected
  [greedy_coreset] 10500/18943 selected
  [greedy_coreset] 11000/18943 selected
  [greedy_cores

## 11. Run 2 — PatchCore (Wide-ResNet50-2)
메인 베이스라인. 논문 기준 grid AUROC ~0.99.

In [12]:
results_wrn50 = run_experiment('grid_patchcore_wrn50', 'wide_resnet50_2')

Running: grid_patchcore_wrn50  (wide_resnet50_2)
Train images: 185
Downloading: "https://download.pytorch.org/models/wide_resnet50_2-9ba9bcbe.pth" to /root/.cache/torch/hub/checkpoints/wide_resnet50_2-9ba9bcbe.pth


100%|██████████| 263M/263M [00:01<00:00, 172MB/s]


[PatchCore] Extracting features from normal images ...
[PatchCore] total_patches=189440  target_coreset=18944  method=greedy
  [greedy_coreset] 500/18943 selected
  [greedy_coreset] 1000/18943 selected
  [greedy_coreset] 1500/18943 selected
  [greedy_coreset] 2000/18943 selected
  [greedy_coreset] 2500/18943 selected
  [greedy_coreset] 3000/18943 selected
  [greedy_coreset] 3500/18943 selected
  [greedy_coreset] 4000/18943 selected
  [greedy_coreset] 4500/18943 selected
  [greedy_coreset] 5000/18943 selected
  [greedy_coreset] 5500/18943 selected
  [greedy_coreset] 6000/18943 selected
  [greedy_coreset] 6500/18943 selected
  [greedy_coreset] 7000/18943 selected
  [greedy_coreset] 7500/18943 selected
  [greedy_coreset] 8000/18943 selected
  [greedy_coreset] 8500/18943 selected
  [greedy_coreset] 9000/18943 selected
  [greedy_coreset] 9500/18943 selected
  [greedy_coreset] 10000/18943 selected
  [greedy_coreset] 10500/18943 selected
  [greedy_coreset] 11000/18943 selected
  [greedy_cores

## 12. 결과 요약

In [13]:
rows = []
for run_tag, res in [('patchcore_r18', results_r18),
                     ('patchcore_wrn50', results_wrn50)]:
    for split, metrics in res.items():
        rows.append({'run': run_tag, 'split': split, **metrics})

summary = pd.DataFrame(rows)
summary_path = REPORTS_DIR / 'grid_baseline_summary.csv'
summary.to_csv(summary_path, index=False)
print(summary.to_string(index=False))
print(f'\nSaved -> {summary_path}')

            run split    auroc    auprc       f1   recall  precision  threshold
  patchcore_r18   val 0.689815 0.624352 0.625000 0.555556   0.714286   2.275638
  patchcore_r18  test 0.811111 0.753825 0.753623 0.866667   0.666667   2.148501
patchcore_wrn50   val 0.957407 0.962547 0.923077 0.888889   0.960000  19.236799
patchcore_wrn50  test 0.938462 0.941642 0.888889 0.800000   1.000000  19.598534

Saved -> /content/drive/MyDrive/SWcapstone/reports/grid_baseline_summary.csv


## 13. 다음 단계

- 위 결과로 **PatchCore 베이스라인 숫자 확정** → 서경님 PaDiM 결과와 같은 테이블에 합쳐 비교
- 확장:
  - 동일 split으로 **EfficientAD / FastFlow** 등 다른 계열 베이스라인 추가 (시간 단축 스토리)
  - 추가학습 기능 (100장 3:1 split 시나리오)
  - 다른 카테고리 (tile, bottle 등) 확장